In [4]:
!pip -q install pandas==2.2.3 scikit-learn joblib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 100.2 MB/s eta 0:00:00


In [1]:
import pandas as pd
import sklearn
import joblib

print("Pandas:", pd.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Joblib:", joblib.__version__)

Pandas: 2.2.3
Scikit-learn: 1.9.0
Joblib: 1.5.3


In [6]:
import json
import zipfile
import joblib
import pandas as pd
import sklearn

from sklearn.compose import ColumnTransformer
from sklearn.datasets import fetch_openml
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

print("Pandas:", pd.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Librerías cargadas correctamente")

Pandas: 2.2.3
Scikit-learn: 1.9.0
Librerías cargadas correctamente


In [7]:
datos_openml = fetch_openml(data_id=42165, as_frame=True)
df = datos_openml.frame.copy()

print("Filas:", df.shape[0])
print("Columnas:", df.shape[1])

df.head()

Filas: 1460
Columnas: 81


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [8]:
columnas = ['Neighborhood', 'GrLivArea', 'YearBuilt', 'YrSold', 'SalePrice']

df_modelo = df[columnas].copy()

print("Columnas seleccionadas:")
print(df_modelo.columns.tolist())

df_modelo.head()

Columnas seleccionadas:
['Neighborhood', 'GrLivArea', 'YearBuilt', 'YrSold', 'SalePrice']


,Neighborhood,GrLivArea,YearBuilt,YrSold,SalePrice
0,CollgCr,1710,2003,2008,208500
1,Veenker,1262,1976,2007,181500
2,CollgCr,1786,2001,2008,223500
3,Crawfor,1717,1915,2006,140000
4,NoRidge,2198,2000,2008,250000


In [9]:
for columna in ['GrLivArea', 'YearBuilt', 'YrSold', 'SalePrice']:
    df_modelo[columna] = pd.to_numeric(df_modelo[columna], errors='coerce')

df_modelo = df_modelo.dropna(subset=columnas).copy()

df_modelo['Area_m2'] = df_modelo['GrLivArea'] * 0.092903

df_modelo['Antiguedad'] = df_modelo['YrSold'] - df_modelo['YearBuilt']

df_modelo = df_modelo[df_modelo['Antiguedad'] >= 0].copy()

print("Filas después de la limpieza:", len(df_modelo))

df_modelo[['Neighborhood', 'Area_m2', 'Antiguedad', 'SalePrice']].head()

Filas después de la limpieza: 1460


,Neighborhood,Area_m2,Antiguedad,SalePrice
0,CollgCr,158.864130,5,208500
1,Veenker,117.243586,31,181500
2,CollgCr,165.924758,7,223500
3,Crawfor,159.514451,91,140000
4,NoRidge,204.200794,8,250000


In [10]:
X = df_modelo[['Neighborhood', 'Area_m2', 'Antiguedad']].copy()
y = df_modelo['SalePrice'].copy()

print("Forma de X:", X.shape)
print("Forma de y:", y.shape)

print("\nPrimeras filas de X:")
display(X.head())

print("\nPrimeros valores de y:")
display(y.head())

Forma de X: (1460, 3)
Forma de y: (1460,)

Primeras filas de X:


,Neighborhood,Area_m2,Antiguedad
0,CollgCr,158.864130,5
1,Veenker,117.243586,31
2,CollgCr,165.924758,7
3,Crawfor,159.514451,91
4,NoRidge,204.200794,8



Primeros valores de y:


,SalePrice
0,208500
1,181500
2,223500
3,140000
4,250000


In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Datos de entrenamiento:", X_train.shape[0])
print("Datos de prueba:", X_test.shape[0])

Datos de entrenamiento: 1168
Datos de prueba: 292


In [12]:
preprocesador = ColumnTransformer(
    transformers=[
        ('sector', OneHotEncoder(handle_unknown='ignore'), ['Neighborhood']),
        ('numericas', 'passthrough', ['Area_m2', 'Antiguedad'])
    ]
)

print("Preprocesador creado correctamente")

Preprocesador creado correctamente


In [13]:
random_forest = RandomForestRegressor(
    n_estimators=80,
    max_depth=10,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

modelo = Pipeline(steps=[
    ('preprocesamiento', preprocesador),
    ('random_forest', random_forest)
])

print("Pipeline creado correctamente")

Pipeline creado correctamente


In [14]:
modelo.fit(X_train, y_train)

print("Modelo entrenado correctamente")

Modelo entrenado correctamente


In [15]:
## EVALUACION DEL MODELO
predicciones = modelo.predict(X_test)

mae = mean_absolute_error(y_test, predicciones)
rmse = mean_squared_error(y_test, predicciones) ** 0.5
r2 = r2_score(y_test, predicciones)

print(f"MAE: ${mae:,.2f}")
print(f"RMSE: ${rmse:,.2f}")
print(f"R²: {r2:.3f}")

MAE: $24,043.85
RMSE: $36,414.53
R²: 0.827


In [16]:
## PREDICCION MANUAL
ejemplo = pd.DataFrame([{
    'Neighborhood': 'OldTown',
    'Area_m2': 120,
    'Antiguedad': 25
}])

precio_estimado = modelo.predict(ejemplo)[0]

print(f"Precio estimado: ${precio_estimado:,.0f}")

Precio estimado: $171,393


In [17]:
## Sectores Validos
sectores_ames = sorted(
    df_modelo['Neighborhood'].astype(str).unique().tolist()
)

print("Cantidad de sectores:", len(sectores_ames))
print(sectores_ames)

Cantidad de sectores: 25
['Blmngtn', 'Blueste', 'BrDale', 'BrkSide', 'ClearCr', 'CollgCr', 'Crawfor', 'Edwards', 'Gilbert', 'IDOTRR', 'MeadowV', 'Mitchel', 'NAmes', 'NPkVill', 'NWAmes', 'NoRidge', 'NridgHt', 'OldTown', 'SWISU', 'Sawyer', 'SawyerW', 'Somerst', 'StoneBr', 'Timber', 'Veenker']


In [18]:
joblib.dump(modelo, 'modelo_casas.pkl')

with open('sectores_ames.json', 'w', encoding='utf-8') as archivo:
    json.dump(sectores_ames, archivo, ensure_ascii=False, indent=2)

with open('requirements_local.txt', 'w', encoding='utf-8') as archivo:
    archivo.write(f'scikit-learn=={sklearn.__version__}\n')
    archivo.write(f'pandas=={pd.__version__}\n')
    archivo.write('joblib\n')
    archivo.write('requests\n')

print("Archivos creados correctamente")

Archivos creados correctamente


In [19]:
import os

for archivo in ['modelo_casas.pkl', 'sectores_ames.json', 'requirements_local.txt']:
    print(archivo, "->", os.path.exists(archivo))

modelo_casas.pkl -> True
sectores_ames.json -> True
requirements_local.txt -> True


In [20]:
from google.colab import files

files.download('modelo_casas.pkl')
files.download('sectores_ames.json')
files.download('requirements_local.txt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>